# ADPr-LLaMA Model Training

Fine-tune `GreatCaptainNemo/ProLLaMA_Stage_1` with a LoRA adapter to predict ADP-ribosylation (ADPr) PTM sites from short peptide sequences. Output format is `Sites=<R5,D12,...>`.

Implemented with `transformers`, `peft`, and `trl.SFTTrainer`. The notebook is intended to run in Google Colab on a GPU runtime (A100 preferred; L4 and T4 also work). The merged model is pushed to the Hugging Face Hub for the evaluation notebook to consume.

## 1. Environment Setup

Install the required libraries. Major versions are pinned for reproducibility. If Colab prompts for a runtime restart (`Runtime > Restart runtime`), do so and resume from the next cell.

In [ ]:
!pip install -q -U \
    "transformers>=4.44,<4.50" \
    "peft>=0.11" \
    "trl>=0.9,<0.12" \
    "accelerate>=0.30" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    "huggingface_hub>=0.24" \
    scikit-learn matplotlib "pandas<3" sentencepiece

# peft >=0.11 errors out if torchao is installed at an incompatible version.
# Colab ships torchao==0.10.0 pre-installed; we don't use it, so remove it.
!pip uninstall -y -q torchao

In [ ]:
import os, json, math, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
)
from peft import LoraConfig, AutoPeftModelForCausalLM
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from huggingface_hub import HfApi, login

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configuration

All hyperparameters are surfaced in this cell: LoRA r=64 / alpha=128, learning rate 3e-4 with cosine decay, per-device batch 16 × gradient accumulation 8 (effective batch 128), up to 8 epochs.

The training data is positives-only (every row in `train.csv` has `has_ptm == 1`), following the methodology established by prior state-of-the-art ADPr-site prediction work: when negative windows are included, models in this family fail to learn site localization, so positives are intentionally over-represented. A consequence is that the model tends to predict at least one site in almost every 21-residue window. The complete inference architecture (defined in the evaluation notebook) compensates for this: overlapping 21-residue windows (stride 5) are aggregated into a per-residue consensus score, and a single F1-optimal threshold derived on a held-out calibration set is applied as the decision rule.

In [ ]:
BASE_MODEL = 'GreatCaptainNemo/ProLLaMA_Stage_1'

TRAIN_CSV = 'datasets/train.csv'
VAL_SIZE = 0.10
RANDOM_SEED = 42

INSTRUCTION = '[Predict the ADP Ribosylation sites given the peptide sequence]'

OUTPUT_DIR = 'saves/adpr-llama'
MERGED_DIR = OUTPUT_DIR + '-merged'

HF_REPO_ID = 'jbenbudd/adpr-llama'

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'down_proj', 'up_proj',
]

LEARNING_RATE = 3e-4
NUM_TRAIN_EPOCHS = 8
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 8
LR_SCHEDULER = 'cosine'
WARMUP_STEPS = 40
MAX_SEQ_LENGTH = 2048
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 5
SAVE_TOTAL_LIMIT = 3
EARLY_STOPPING_PATIENCE = 3
MAX_GRAD_NORM = 1.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Authenticate to Hugging Face

In Colab, store your token in `Secrets > HF_API_TOKEN` (the lock icon in the left sidebar). Outside Colab, export `HF_API_TOKEN` in your shell.

In [ ]:
try:
    from google.colab import userdata, drive
    HF_TOKEN = userdata.get('HF_API_TOKEN')
    # Mount Drive only if the CSVs are stored there; uncomment to use:
    # drive.mount('/content/drive', force_remount=False)
    # TRAIN_CSV = '/content/drive/MyDrive/adpr-llama/datasets/train.csv'
except:
    HF_TOKEN = os.environ.get('HF_API_TOKEN')

assert HF_TOKEN, 'No HF token found. Set HF_API_TOKEN in Colab Secrets or env vars.'
login(token=HF_TOKEN)
print('Logged in to Hugging Face.')

## 4. Data Preparation

Each row of `train.csv` is a 21-residue peptide window with `has_ptm == 1` and is converted into an Alpaca-style instruction record:

```
Below is an instruction that describes a task. ...

### Instruction:
[Predict the ADP Ribosylation sites given the peptide sequence]
Seq=<RGGFGRGGGRGGFNKGQDQGP>

### Response:
Sites=<R1,S4>
```

The `### Response:\n` marker is used by `DataCollatorForCompletionOnlyLM` to mask everything before the response with `-100`, so loss is computed only on the answer tokens. Because SentencePiece tokenization is context-dependent, the marker is passed to the collator as a sequence of token IDs derived from in-context encoding rather than as a raw string.

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print(f'Loaded {len(df):,} rows from {TRAIN_CSV}')
print('Columns:', list(df.columns))
df[['Uniprot_ID', 'seq_sequence', 'sites_in_window', 'has_ptm']].head()

In [ ]:
df = df.dropna(subset=['seq_sequence', 'sites_in_window']).copy()
df['sites_in_window'] = df['sites_in_window'].astype(str).str.replace(' ', '', regex=False)

def to_record(row):
    return {
        'instruction': INSTRUCTION,
        'input':  f'Seq=<{row.seq_sequence}>',
        'output': f'Sites=<{row.sites_in_window}>',
        'uniprot_id': row.Uniprot_ID,
    }

records = [to_record(r) for _, r in df.iterrows()]
print(f'Built {len(records):,} records.')
print('Sample record:')
print(json.dumps(records[0], indent=2))

### Group-split by `Uniprot_ID`

The training set is constructed by sliding a 21-residue window across each protein with stride 5, so many rows derive from the same protein at overlapping positions. A naive random split would place windows from the same protein in both partitions, producing optimistically biased `eval_loss` values through information leakage. `GroupShuffleSplit` keyed on `Uniprot_ID` keeps all windows from a given protein in a single partition.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
groups = [r['uniprot_id'] for r in records]
train_idx, val_idx = next(gss.split(records, groups=groups))
train_records = [records[i] for i in train_idx]
val_records   = [records[i] for i in val_idx]

train_proteins = {r['uniprot_id'] for r in train_records}
val_proteins   = {r['uniprot_id'] for r in val_records}
leak = train_proteins & val_proteins
print(f'Train:   {len(train_records):,} windows | {len(train_proteins):,} proteins')
print(f'Val:     {len(val_records):,} windows | {len(val_proteins):,} proteins')
print(f'Overlap: {len(leak)} proteins (must be 0)')
assert not leak, 'Group split leaked proteins across train/val'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = 'right'

ALPACA_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n{input}\n\n'
    '### Response:\n{output}'
)

def to_text(rec):
    return ALPACA_TEMPLATE.format(**rec) + tokenizer.eos_token

train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_records])
val_ds   = Dataset.from_list([{'text': to_text(r)} for r in val_records])

print('Example training text:')
print('-' * 60)
print(train_ds[0]['text'])
print('-' * 60)

## 5. Load Model and Configure LoRA

The base model is loaded in bfloat16 with `device_map='auto'` and gradient checkpointing enabled. The LoRA configuration targets the attention projections (`q`, `k`, `v`, `o`) and the MLP projections (`gate`, `down`, `up`).

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)
print(peft_config)

## 6. Loss Curves and Early Stopping

**Training loss vs. evaluation loss.** Training loss is computed on the partition the model is actively updated on. Evaluation loss is computed on held-out windows (`val_ds`) the model does not see during gradient updates and is therefore the operative measure of generalization.

**Interpreting the curves.**
- Both losses falling together indicates healthy learning.
- Training loss falling while evaluation loss plateaus indicates the model has begun to memorize the training set.
- Training loss falling while evaluation loss rises indicates overfitting; the checkpoint with the lowest `eval_loss` is the one to retain.
- Jaggedness in `eval_loss` is expected with small validation sets; trends across multiple evaluation steps are more informative than single points.

**Early stopping.** `EarlyStoppingCallback(patience=3)` terminates training if `eval_loss` fails to improve for three consecutive evaluations. Combined with `load_best_model_at_end=True`, the final saved model is the checkpoint with the lowest `eval_loss` observed during training.

**Learning rate schedule.** Linear warmup over 40 steps avoids large early updates that destabilize LoRA-attached layers; the subsequent cosine decay reduces the learning rate toward the end of training so that late updates refine the loss surface rather than oscillate across it.

**Training distribution.** The training set is restricted to windows containing at least one PTM (`has_ptm == 1`), following the precedent established by prior state-of-the-art ADPr-site prediction work: when negative windows are included, models in this family fail to learn site localization. Positives are intentionally over-represented, and the model consequently tends to predict at least one site in almost every window. This is handled at inference (in the evaluation notebook): overlapping 21-residue windows (stride 5) are aggregated into a per-residue consensus score, and a single F1-optimal threshold derived on a held-out calibration set is applied as the decision rule. `eval_loss` remains the appropriate stopping signal during training because it measures how well the model localizes sites within windows known to contain at least one site.

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    bf16=True,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='adamw_torch',
    report_to='none',
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    gradient_checkpointing=True,
    seed=RANDOM_SEED,
)

response_template = '### Response:\n'
response_template_ids = tokenizer.encode('\n' + response_template, add_special_tokens=False)[2:]
print('response_template_ids:', response_template_ids)
print('decoded back        :', repr(tokenizer.decode(response_template_ids)))
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    args=sft_config,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

trainer.model.print_trainable_parameters()

## 7. Train

In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, 'trainer_state.json'), 'w') as f:
    json.dump(trainer.state.__dict__, f, indent=2, default=str)
with open(os.path.join(OUTPUT_DIR, 'trainer_log_history.json'), 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)

print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best eval loss:', trainer.state.best_metric)

## 8. Plot Loss Curves

Training and evaluation loss curves are saved as a PNG for inclusion in the model card.

In [ ]:
log_history = trainer.state.log_history
train_logs = [(l['step'], l['loss']) for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs  = [(l['step'], l['eval_loss']) for l in log_history if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(9, 5))
if train_logs:
    s, v = zip(*train_logs)
    ax.plot(s, v, label='Train loss', alpha=0.7, linewidth=1.2)
if eval_logs:
    s, v = zip(*eval_logs)
    ax.plot(s, v, label='Eval loss', marker='o', linewidth=2)
if trainer.state.best_metric is not None and eval_logs:
    best_step = min(eval_logs, key=lambda x: x[1])[0]
    ax.axvline(best_step, color='gray', linestyle='--', alpha=0.5,
               label=f'Best checkpoint (step {best_step})')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
loss_plot_path = os.path.join(OUTPUT_DIR, 'training_loss.png')
fig.savefig(loss_plot_path, dpi=150)
plt.show()
print('Saved:', loss_plot_path)

## 9. Sample Inference

Generate a completion from the trained adapter to confirm it produces output in the expected `Sites=<...>` format.

In [ ]:
model.eval()
sample = val_records[0]
prompt = ALPACA_TEMPLATE.format(instruction=sample['instruction'], input=sample['input'], output='')
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False, pad_token_id=tokenizer.pad_token_id)
completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('Input :', sample['input'])
print('Expect:', sample['output'])
print('Got   :', completion.strip())

## 10. Merge LoRA Adapter into the Base Model

We merge the LoRA weights back into the base model in fp16. This produces a single self-contained model that the evaluation notebook can load with one call to `AutoModelForCausalLM.from_pretrained`.

In [ ]:
del model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

merged = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    device_map='auto',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
tokenizer.save_pretrained(MERGED_DIR)
print('Merged model saved to', MERGED_DIR)
!ls -lh {MERGED_DIR}

## 11. Push the Merged Model to Hugging Face

This cell uploads the merged weights, tokenizer, loss plot, and trainer logs. The evaluation notebook overwrites `README.md` after computing test metrics.

In [ ]:
import shutil

for fn in ['training_loss.png', 'trainer_log_history.json', 'trainer_state.json']:
    src = os.path.join(OUTPUT_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, MERGED_DIR)

stub_card = f'''---
base_model: {BASE_MODEL}
tags:
  - protein
  - ptm
  - adp-ribosylation
  - lora
  - peft
library_name: transformers
---

# ADPr-LLaMA

LoRA-fine-tuned `{BASE_MODEL}` for predicting ADP-ribosylation (ADPr) PTM sites from 21-residue peptide windows. Output format: `Sites=<R5,D12,...>`.

This is a **training-only stub card**. Final metrics (ROC, accuracy, precision, recall, F1, confusion matrix) are filled in by the companion evaluation notebook after running on the held-out test set.
'''
with open(os.path.join(MERGED_DIR, 'README.md'), 'w') as f:
    f.write(stub_card)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=False)
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
    delete_patterns='*',
    commit_message='Push ADPr-LLaMA training artifacts',
)
print('Pushed to', f'https://huggingface.co/{HF_REPO_ID}')

## Next Steps

Proceed to the evaluation notebook (`evaluation/evaluate_adpr_llama.ipynb`) to compute test metrics and finalize the model card.